In [1]:
import numpy as np
import pandas as pd


In [ ]:
def limpiar_base_datos(ruta_archivo):
  # 1. Carga de datos
  # Si usas Google Colab puedes mantener files.upload(), o leer directo la ruta
  df = pd.read_csv(ruta_archivo)

  print(f"Dimensiones iniciales: {df.shape}")

  # ==========================================
  # 2. LIMPIEZA DE COLUMNAS DE TEXTO (Strings)
  # ==========================================
  # Ejemplo para columnas de texto categórico o nombres:
  columnas_texto = ["nombre", "pais_origen", "categoria_peso"]

  for col in columnas_texto:
    if col in df.columns:
      # Rellenar nulos, quitar espacios extra, capitalizar (Ej: "juan perez" -> "Juan Perez")
      df[col] = (
          df[col].fillna("Desconocido").astype(str).str.strip().str.title()
      )

  # Diccionarios de mapeo personalizados (opcional para estandarizar categorías)
  # pais_map = {"Mex": "México", "USA": "Estados Unidos"}
  # df['pais_origen'] = df['pais_origen'].replace(pais_map)

  # ==========================================
  # 3. LIMPIEZA DE VARIABLES CATEGÓRICAS / MAPEOS
  # ==========================================
  if "guardia" in df.columns:
    guardia_map = {"ortodoxo": "Ortodoxo", "ZURDO": "Zurdo", "Ambi": "Ambidiestro"}
    df["guardia"] = (
        df["guardia"]
        .map(lambda x: guardia_map.get(x, x))
        .fillna("Desconocida")
        .str.strip()
    )

  # ==========================================
  # 4. LIMPIEZA DE VARIABLES NUMÉRICAS (Enteros)
  # ==========================================
  # Ejemplo: Victorias, derrotas, empates, edades, etc.
  columnas_enteras = ["victorias", "derrotas", "empates"]

  for col in columnas_enteras:
    if col in df.columns:
      df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).abs().astype(int)

  # ==========================================
  # 5. LIMPIEZA DE VARIABLES NUMÉRICAS (Decimales / Continuas)
  # ==========================================
  # Ejemplo para pesos, precios, medidas:
  if "peso_kg" in df.columns:
    df["peso_kg"] = pd.to_numeric(df["peso_kg"], errors="coerce").abs()
    # Corrección lógica de valores atípicos por error de dedo (ej. multiplicados por 10)
    df.loc[df["peso_kg"] > 250, "peso_kg"] = df["peso_kg"] / 10
    df["peso_kg"] = df["peso_kg"].round(1)

  # Ejemplo para estaturas o dimensiones (convertir metros a cm si fuera necesario, etc.)
  if "estatura_cm" in df.columns:
    df["estatura_cm"] = pd.to_numeric(df["estatura_cm"], errors="coerce").abs()
    # Lógica de corrección de escala
    df.loc[df["estatura_cm"] < 3, "estatura_cm"] = (
        df["estatura_cm"] * 100
    )  # Metros a cm
    df["estatura_cm"] = df["estatura_cm"].round(0).astype(int)

  # ==========================================
  # 6. LIMPIEZA DE FORMATOS MONETARIOS
  # ==========================================
  if "ganancias_usd" in df.columns:
    df["ganancias_usd"] = (
        df["ganancias_usd"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("€", "", regex=False)  # Por si hay otras monedas
        .str.strip()
    )
    df["ganancias_usd"] = pd.to_numeric(df["ganancias_usd"], errors="coerce")

  # ==========================================
  # 7. LIMPIEZA DE FECHAS
  # ==========================================
  if "fecha_debut" in df.columns:
    df["fecha_debut"] = pd.to_datetime(
        df["fecha_debut"], format="mixed", errors="coerce"
    ).dt.strftime("%Y-%m-%d")

  print(f"Dimensiones finales: {df.shape}")
  return df


# --- CÓMO USARLO ---
# df_limpio = limpiar_base_datos("tu_archivo.csv")
# df_limpio.to_csv("archivo_limpio.csv", index=False)